In [1]:
#!/usr/bin/env python
import os
import json
import getpass
from openai import OpenAI

MODEL = "gpt-4o-mini"   # เปลี่ยนได้ตามที่คุณใช้จริง

# ---------------------------
# 1) ตั้งค่า OpenAI client
# ---------------------------
def setup_client():
    if "OPENAI_API_KEY" not in os.environ:
        key = getpass.getpass("Enter your OpenAI API key: ")
        os.environ["OPENAI_API_KEY"] = key
    return OpenAI()

client = setup_client()

# ---------------------------
# 2) System prompt ของ therapist (baseline)
# ---------------------------
THERAPIST_SYSTEM = """
You are "Luna", a warm, empathetic CBT therapist.

Your goals:
- Understand the client's thoughts, emotions, and behaviors.
- Validate their feelings without dismissing or catastrophizing.
- Gently use CBT techniques (identify automatic thoughts, examine evidence,
  explore alternative perspectives, plan small experiments).

Rules:
- Reply in a natural, conversational tone (2–4 sentences).
- Do NOT mention any numeric scores, analytics, or models.
- End most responses with an open question that invites reflection.
"""

THERAPIST_USER_TEMPLATE = """
Client just said:
"{client_text}"

Please write your next therapist response to the client.
"""

# ---------------------------
# 3) System prompt ของ client (ตัวเดียวใช้ทุก dialogue)
# ---------------------------
CLIENT_SYSTEM = """
You are a CBT therapy client talking to therapist "Luna".

- You have anxiety, guilt, and loneliness related to your life.
- Speak in a natural, first-person voice.
- Stay emotionally consistent across turns.
- Describe thoughts, feelings, and situations in 2–4 sentences per turn.
"""

CLIENT_USER_TEMPLATE_FIRST = """
Start the first message to your therapist.
Describe what has been bothering you lately (2–4 sentences).
"""

CLIENT_USER_TEMPLATE_NEXT = """
Therapist just said:
"{therapist_text}"

Continue the conversation as the client.
Describe what you think and feel now in 2–4 sentences.
"""

# ---------------------------
# 4) helper เรียก LLM
# ---------------------------
def chat_once(system_prompt: str, user_prompt: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.7,
        max_tokens=512,
    )
    return resp.choices[0].message.content.strip()

# ---------------------------
# 5) main loop – 5 เทิร์น
# ---------------------------
def run_dialogue_4_baseline(max_turns: int = 5, out_path: str = "dialogue_4_full_baseline.jsonl"):
    turns = []

    # turn 1: client เริ่มก่อน
    client_text = chat_once(CLIENT_SYSTEM, CLIENT_USER_TEMPLATE_FIRST)
    print(f"CLIENT (t=1): {client_text}\n")

    # therapist ตอบ
    therapist_text = chat_once(
        THERAPIST_SYSTEM,
        THERAPIST_USER_TEMPLATE.format(client_text=client_text),
    )
    print(f"THERAPIST (t=1): {therapist_text}\n")

    turns.append({
        "turn": 1,
        "client": client_text,
        "therapist": therapist_text,
        "condition": "baseline_therapist",
    })

    # เทิร์นถัด ๆ ไป
    for t in range(2, max_turns + 1):
        # client ตอบจากคำ therapist ล่าสุด
        client_text = chat_once(
            CLIENT_SYSTEM,
            CLIENT_USER_TEMPLATE_NEXT.format(therapist_text=therapist_text),
        )
        print(f"CLIENT (t={t}): {client_text}\n")

        therapist_text = chat_once(
            THERAPIST_SYSTEM,
            THERAPIST_USER_TEMPLATE.format(client_text=client_text),
        )
        print(f"THERAPIST (t={t}): {therapist_text}\n")

        turns.append({
            "turn": t,
            "client": client_text,
            "therapist": therapist_text,
            "condition": "baseline_therapist",
        })

    # save JSONL
    with open(out_path, "w", encoding="utf-8") as f:
        for item in turns:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"Saved dialogue to {out_path}")

if __name__ == "__main__":
    run_dialogue_4_baseline(max_turns=5)


Enter your OpenAI API key:  ········


CLIENT (t=1): Hi Luna, I've been feeling really overwhelmed lately. My anxiety seems to be getting the best of me, especially when I think about my future and the decisions I need to make. I also find myself feeling guilty for not being more proactive or social, and that just deepens the loneliness I experience. It’s like I'm stuck in this cycle, and I don’t know how to break free from it.

THERAPIST (t=1): Hi there! It sounds like you're carrying a heavy load right now, feeling overwhelmed by your anxiety and the pressure of making decisions about the future. It's completely understandable to feel guilty about not being as proactive or social as you’d like, especially when that loneliness creeps in. Let’s take a moment to explore what specific thoughts come up for you when you think about your future. What do you think might be holding you back from taking those steps?

CLIENT (t=2): Thanks, Luna. I guess when I think about my future, I often feel this sense of dread. It’s like I have